In [ ]:
# Day 23 - LLM Agents & Tool Use
!pip install -q langchain langchain-community langchain-huggingface

from langchain_huggingface import HuggingFacePipeline
from langchain.agents import load_tools, initialize_agent, AgentType
from langchain.tools import Tool
import torch
from transformers import pipeline

In [ ]:
# Load LLM
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=200,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)
print("✅ LLM Loaded!")

In [ ]:
# Define Tools
def calculator(expression):
    try:
        return str(eval(expression))
    except:
        return "Error in calculation"

tools = [
    Tool(
        name="Calculator",
        func=calculator,
        description="Useful for doing math calculations"
    ),
    Tool(
        name="General Knowledge",
        func=lambda x: f"I don't have real-time knowledge but I can help with general questions about Ethiopia and AI.",
        description="Use this for general knowledge questions"
    )
]

In [ ]:
# Initialize Agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print("✅ Agent Ready!")

In [ ]:
# Test the Agent
print(agent.run("What is 15 * 27?"))
print("\n--- Next Question ---")
print(agent.run("Tell me something about Ethiopia."))